# Training OWN tokenizer for URDU

In [1]:
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

# To give vector embedding and token embedding
import torch
import torch.nn as nn



c:\Users\Axil\Desktop\ScratchLLM\Baat Urdu_l4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Streaming the dataset to avoid loading it all into memory at once
print("Loading dataset...")
dataset = load_dataset("allenai/c4", "ur",split="train", streaming=True)

Loading dataset...


In [4]:


# Intialize a BPE tokenizer
tokenizer = Tokenizer(BPE(unk_token="[unk]"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

#Configure the tokenizer trainer
trainer=BpeTrainer(vocab_size=32000,special_tokens=["[unk]","[pad]","[endoftext]"],show_progress=True)



In [7]:
#Feeding text to trainer in batches

def batch_iterator(batch_size=1000,max_docs=500000):
    batch = []

    for i, doc in enumerate(dataset):
        if i >= max_docs:
            break
        batch.append(doc["text"])
        if len(batch)==batch_size:
            yield batch
            batch=[]
    if batch:
        yield batch
        

In [9]:
# Train the tokenizer and export
print("Training tokenizer...")
tokenizer.train_from_iterator(batch_iterator(),trainer=trainer)
tokenizer.save("tokenizer.json")
print("tokenizer saved to tokenizer.json", "vocab size:", tokenizer.get_vocab_size())

Training tokenizer...
tokenizer saved to tokenizer.json vocab size: 32000


In [11]:
# test the tokenizer
text= "یہ ایک ٹیسٹ ہے۔"
tokenizer.encode(text)
print("Encoded text:", tokenizer.encode(text).tokens)
text= "This is a test"
tokenizer.encode(text)
print("Encoded text:", tokenizer.encode(text).tokens)


Encoded text: ['ÛĮÛģ', 'ĠØ§ÛĮÚ©', 'ĠÙ¹ÛĮØ³Ù¹', 'ĠÛģÛĴ', 'ÛĶ']
Encoded text: ['This', 'Ġis', 'Ġa', 'Ġtest']
